# arXiv: sample collection, 2022–2026

Retrieve 20 records first submitted from January 1, 2022 through October 31, 2026 (UTC). Inspect titles, abstracts, and metadata before cleaning or expanding collection. Restart the kernel before running this revised notebook.

This is a retrieval check, **not a representative sample**: the earliest submissions are returned first. The query covers the agreed categories, including cross-listings. Robotics (cs.RO) and information retrieval (cs.IR) are included. Application papers are retained for later classification.

The API returns current metadata, which may include revisions made after the submission cutoff. No PDFs or full texts are downloaded, and no complete-corpus collection is implemented.

Reference: [arXiv API manual](https://info.arxiv.org/help/api/user-manual.html). Requires Python, pandas, and an IPython/Jupyter kernel. The collection module and offline tests use only Python's standard library.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Support starting Jupyter in either the repository root or notebooks/.
project_root = Path.cwd()
if not (project_root / "arxiv_collection.py").is_file():
    project_root = project_root.parent
if not (project_root / "arxiv_collection.py").is_file():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from arxiv_collection import build_url, fetch_xml, parse_feed

CATEGORIES = ["cs.AI", "cs.LG", "stat.ML", "cs.CL", "cs.CV", "cs.MA", "cs.NE", "cs.IR", "cs.RO"]
START = "202201010000"
END = "202610312359"
SAMPLE_SIZE = 20

api_url = build_url(CATEGORIES, START, END, max_results=SAMPLE_SIZE)
print("Categories:", CATEGORIES)
print("Submission range (UTC, YYYYMMDDHHMM):", START, "to", END)
print("URL:", api_url)


## Retrieve one page

This cell makes one request. Allow at least three seconds between requests if rerunning it. Network or API failures are raised rather than replaced with example data.


In [ ]:
raw_xml = fetch_xml(api_url)
retrieved_at = datetime.now(timezone.utc).isoformat()
feed = parse_feed(raw_xml)
records = feed["records"]

print("Retrieved at (UTC):", retrieved_at)
print("Matching records reported by API:", feed["total_results"])
print("Records in this page:", len(records))
if not records:
    raise RuntimeError("The API returned no records. Inspect the query before proceeding.")


In [ ]:
df = pd.DataFrame(records)
# Keep returned text and date strings unchanged for inspection.
with pd.option_context("display.max_rows", SAMPLE_SIZE, "display.max_colwidth", None):
    display(df[["arxiv_id", "title", "abstract", "submitted_at", "updated_at", "primary_category", "categories"]])


## Inspect quality without changing the data

Check missing fields, duplicate paper IDs (ignoring version suffixes), submission dates, and category matches. Counts apply only to this page. Dates below are parsed into temporary series; the original table is unchanged.


In [ ]:
required = ["arxiv_id", "title", "abstract", "submitted_at", "primary_category"]
missing = pd.Series({column: int(df[column].fillna("").str.strip().eq("").sum()) for column in required}, name="missing_or_blank")
display(missing.to_frame())

submitted = pd.to_datetime(df["submitted_at"], utc=True, errors="coerce")
base_ids = df["arxiv_id"].str.replace(r"v\d+$", "", regex=True)
# Use the same settings as the API query; END includes the entire last minute.
start_bound = pd.to_datetime(START, format="%Y%m%d%H%M", utc=True)
end_exclusive = pd.to_datetime(END, format="%Y%m%d%H%M", utc=True) + pd.Timedelta(minutes=1)
in_range = (submitted >= start_bound) & (submitted < end_exclusive)
category_matches = df["categories"].apply(lambda values: bool(set(values) & set(CATEGORIES)))
checks = pd.Series({
    "records": len(df),
    "duplicate_ids_ignoring_version": int(base_ids.duplicated().sum()),
    "invalid_submission_dates": int(submitted.isna().sum()),
    "records_outside_submission_range": int((~in_range & submitted.notna()).sum()),
    "records_without_a_requested_category": int((~category_matches).sum()),
})
display(checks.to_frame("count"))
print("Sample submission range:", submitted.min(), "to", submitted.max())
display(df["primary_category"].value_counts().rename_axis("primary_category").to_frame("sample_records"))


## Count matches before expanding collection

Use the same submission dates and categories as above. Each query requests only one record; `total_results` reports all matches, not the page size. This makes 10 requests: the combined query and nine category queries, with a three-second pause before each.

Category counts include primary and additional categories, so they overlap and **must not be summed**. The combined query counts each matching paper once. These are API-reported counts, not a verified complete download.


In [ ]:
import time

count_rows = []
count_queries = [("Combined (unique papers)", CATEGORIES)] + [
    (category, [category]) for category in CATEGORIES
]
for label, categories in count_queries:
    time.sleep(3)
    count_url = build_url(categories, START, END, max_results=1)
    count_feed = parse_feed(fetch_xml(count_url))
    total = count_feed["total_results"]
    if total is None:
        raise RuntimeError(f"The API did not return a total count for {label}.")
    count_rows.append({"scope": label, "matching_papers": total})
    print(f"{label}: {total:,}", flush=True)

counts_df = pd.DataFrame(count_rows)
print("Counts retrieved at (UTC):", datetime.now(timezone.utc).isoformat())
print("Submission range (UTC, YYYYMMDDHHMM):", START, "to", END)
display(counts_df)


## Review before expanding

Inspect abstracts, categories, and quality checks above. Decide what cleaning is needed. No records have been dropped, text cleaned, or topics inferred.

Run the offline tests yourself from the repository root in a terminal:

```bash
python3 -m unittest discover -s tests -v
```

The tests check URL construction, parsing synthetic responses, and simulated download behavior. They make no API requests. Passing them does not establish complete coverage or validate any scientific conclusions.
